<a href="https://colab.research.google.com/github/omque/CSE291P-Bioinformatics-Project/blob/main/Omair-Data-Clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Drug Targets Project — CSE 291 Spring 2026
## Dataset: MSV000081932 — Target Landscape of Clinical Kinase Inhibitors

## What This Notebook Does

This notebook covers the initial preprocessing of the raw search results
file from MSV000081932. The raw file is a TSV with 1,033 columns and 83,706
rows, where each row represents a detected peptide variant across all
samples.

The 1,033 columns break down as:
- **33 metadata columns**: descriptive information about each peptide
  (e.g., which protein it came from, its charge state, quality filters)
- **1,000 measurement columns**: 50 drugs × 20 columns each, representing
  the measured peptide abundance at 8 drug concentrations (3nM through
  30000nM), a DMSO vehicle control, and a PDPD control — each paired with
  a corresponding `_unmod` version tracking unmodified peptide abundance

For the purpose of drug target detection, we only need a focused subset of
this data. This preprocessing step filters the file down to **507 columns**:
- 7 metadata columns essential for identifying peptides and proteins
- 500 measurement columns (10 per drug × 50 drugs), keeping only the
  direct abundance measurements and discarding the `_unmod` paired columns

No rows are removed at this stage. The output is a filtered TSV ready for
downstream analysis.

## Environment

This notebook is intended to be run via **Google Colab**, accessed through
the **Colab extension in VSCode**. The code is written in Python and uses
the pandas library for data manipulation.

### Step 1: Imports

We import pandas, the standard Python library for working with tabular
data. It will handle loading, filtering, and saving our TSV file.
Nothing else is needed for this task.

In [1]:
import pandas as pd

### Step 2: Mount Google Drive

This connects your Google Drive to the Colab session, making your files
accessible under the path /content/drive/MyDrive/. You will be prompted
to authorize access the first time you run this. After mounting, the
notebook can read and write files in your Drive just like any normal
folder.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Step 3: Define File Paths

We set the input and output paths in one place so they are easy to update
if files move.

In [3]:
INPUT_PATH = '/content/drive/MyDrive/Spring Quarter 2026/CSE 291P - Big Data Science and Knowledge/Project/Datasets/variants_intensity_complete.tsv'
OUTPUT_PATH = '/content/drive/MyDrive/Spring Quarter 2026/CSE 291P - Big Data Science and Knowledge/Project/Datasets/variants_intensity_filtered.tsv'

### Step 4: Load the TSV File

We read the TSV into a pandas DataFrame. The sep='\t' argument tells
pandas this is tab-separated rather than comma-separated. low_memory=False
is a precaution for wide files like this one — it tells pandas to read the
whole file before deciding on column data types, which avoids warnings.

The print statement confirms we loaded the expected shape: 83,706 rows and
1,033 columns.

In [4]:
df = pd.read_csv(INPUT_PATH, sep='\t', low_memory=False)
print(f"Loaded dataframe with {df.shape[0]} rows and {df.shape[1]} columns")

Loaded dataframe with 83706 rows and 1033 columns


### Step 5: Define Metadata Columns to Keep

We hardcode the 7 metadata columns we need. These describe each peptide
row — what it is, which protein it came from, and basic quality
information. All other metadata columns (26 of them) will be discarded. Data on rationale for discarding are included in research notes.

In [5]:
METADATA_COLS = [
    'Variant',
    'Unmod variant',
    'Charge',
    'Top canonical protein',
    'Canonical proteins',
    'Is Decoy',
    'Variant FDR'
]

print(f"Metadata columns to keep: {len(METADATA_COLS)}")

Metadata columns to keep: 7


### Step 6: Identify Measurement Columns to Keep

From all 1,033 columns, we select only the measurement columns we want by
applying three filters:
- Only keep columns whose names start with '_dyn_#' — this is the prefix
  shared by all genuine measurement columns, and cleanly excludes the 26
  metadata columns we want to discard
- Exclude any column containing 'PDPD' anywhere in its name — these turned
  out to appear mid-name rather than as a suffix
- Exclude any column ending in '_unmod'

This leaves us with the 8 concentration columns (3nM through 30000nM) and
DMSO per drug, for 9 columns × 50 drugs = 450 measurement columns. The
print statement should confirm exactly 450.

In [13]:
DISCARD_SUFFIXES = ('_unmod', 'PDPD')

all_columns = df.columns.tolist()
measurement_cols = [
    col for col in all_columns
    if col not in METADATA_COLS
    and col.startswith('_dyn_#')
    and not col.endswith('_unmod')
    and 'PDPD' not in col
]

print(f"Measurement columns to keep: {len(measurement_cols)}")

Measurement columns to keep: 450


### Step 7: Sanity Check

Before filtering, we verify the column counts add up to the expected 457
(7 metadata + 450 measurement). If you see a different number, it likely
means some column names have unexpected formatting — e.g. 'PDPD' appearing
mid-name rather than as a suffix, or spacing differences. Worth
investigating before proceeding.

In [15]:
expected_total = len(METADATA_COLS) + len(measurement_cols)
print(f"Metadata columns:    {len(METADATA_COLS)}")
print(f"Measurement columns: {len(measurement_cols)}")
print(f"Total to keep:       {expected_total}  (expected 457)")

Metadata columns:    7
Measurement columns: 450
Total to keep:       457  (expected 457)


### Step 8: Filter the DataFrame

We create a new DataFrame containing only our chosen columns. The original
DataFrame df is left untouched. The shape should read (83706, 457).

In [16]:
df_filtered = df[METADATA_COLS + measurement_cols]
print(f"Filtered dataframe shape: {df_filtered.shape}")

Filtered dataframe shape: (83706, 457)


### Step 9: Save to Google Drive

We write the filtered DataFrame back out as a TSV to the output path
defined in Step 3. index=False prevents pandas from adding a row-number
column to the file, which we don't need. The file will appear in your
Drive and persist after the session ends.

In [17]:
df_filtered.to_csv(OUTPUT_PATH, sep='\t', index=False)
print(f"Saved filtered file to: {OUTPUT_PATH}")

Saved filtered file to: /content/drive/MyDrive/Spring Quarter 2026/CSE 291P - Big Data Science and Knowledge/Project/Datasets/variants_intensity_filtered.tsv
